In [1]:
!pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.4 MB/s eta 0:00:00


In [2]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/uom210636r/misato/MD_split_5.hdf5
/kaggle/input/datasets/uom210636r/misato/MD_split_9.hdf5
/kaggle/input/datasets/uom210636r/misato/MD_split_6.hdf5
/kaggle/input/datasets/uom210636r/misato/MD_split_2.hdf5
/kaggle/input/datasets/uom210636r/misato/preprocess_1.hdf5
/kaggle/input/datasets/uom210636r/misato/test_keys.txt
/kaggle/input/datasets/uom210636r/misato/MD_split_10.hdf5
/kaggle/input/datasets/uom210636r/misato/MD_split_1.hdf5
/kaggle/input/datasets/uom210636r/misato/MD_split_7.hdf5
/kaggle/input/datasets/uom210636r/misato/MD_split_3.hdf5
/kaggle/input/datasets/uom210636r/misato/train_keys.txt
/kaggle/input/datasets/uom210636r/misato/val_keys.txt
/kaggle/input/datasets/uom210636r/misato/MD_split_8.hdf5
/kaggle/input/datasets/uom210636r/misato/MD_split_4.hdf5
/kaggle/input/datasets/uom210636r/misato/train_test_val/MD_split_3_val.txt
/kaggle/input/datasets/uom210636r/misato/train_test_val/MD_split_5_train.txt
/kaggle/input/datasets/uom210636r/misato/train_test_v

In [3]:
import os, sys, subprocess

# Clone this repo on Kaggle to get all components (graphs, models, training, etc.)
REPO_URL = "https://github.com/Vaheshan/GraphMD.git"  
TARGET_DIR = "/kaggle/working/project"

if "KAGGLE_KERNEL_RUN_TYPE" in os.environ:
    if not os.path.exists(TARGET_DIR):
        print(f"Cloning repo from {REPO_URL} into {TARGET_DIR} ...")
        subprocess.run(["git", "clone", REPO_URL, TARGET_DIR], check=False)
    if TARGET_DIR not in sys.path:
        sys.path.append(TARGET_DIR)
    print("Repo path added to sys.path:", TARGET_DIR)
else:
    print("Not running on Kaggle, skipping clone.")

Cloning repo from https://github.com/Vaheshan/GraphMD.git into /kaggle/working/project ...


Cloning into '/kaggle/working/project'...


Repo path added to sys.path: /kaggle/working/project


In [4]:
import os
import math
import numpy as np
import h5py
from typing import List, Dict, Any, Tuple

import torch
from torch import Tensor
from torch_geometric.data import Data, Batch

from graphs import (
    ProteinGraphBuilder,
    CorrelationEdgeBuilder,
    PocketGraphBuilder,
    ProteinGraphInputs,
    PocketGraphInputs,
)
from models import MultiscaleMDGNN
from training.batch_utils import collate_complexes
from training.trainer import Trainer

print('Imports done.')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

# -------------------------------------------------------------------
# Configuration (adapt paths for Kaggle)
# -------------------------------------------------------------------
ROOT_DIR = '/kaggle/input/datasets/uom210636r/misato/'  # Thilini

SPLIT_MODE = 'train'  # 'train', 'val', or 'test'
#Thilini
RAW_DIR = ROOT_DIR
SPLIT_CANDIDATES = [
    os.path.join(RAW_DIR, f'{SPLIT_MODE}.txt')
    # os.path.join(RAW_DIR, f'{SPLIT_MODE}_MD.txt'), # Thilini
]

# Collect all MD*.hdf5 files (e.g., MD_0.hdf5, MD_1.hdf5, ..., MD_9.hdf5)
MD_HDF5_FILES = []
if os.path.exists(RAW_DIR):
    for fname in sorted(os.listdir(RAW_DIR)):
        if fname.lower().endswith('.hdf5') and fname.lower().startswith('md'):
            MD_HDF5_FILES.append(os.path.join(RAW_DIR, fname))

print('Found MD HDF5 files:')
for p in MD_HDF5_FILES:
    print('  ', p)

PROCESSED_GRAPH_PATH = '/kaggle/working/misato_md_graphs.pt'
MODEL_PATH = '/kaggle/working/multiscale_mdg_nn_pretrained.pth'
CHECKPOINT_PATH = '/kaggle/working/multiscale_mdg_nn_checkpoint.pth'
LOAD_CHECKPOINT_PATH='/kaggle/working/project/output/multiscale_mdg_nn_checkpoint.pth'

# Graph / training hyperparameters
# Use smaller values by default to fit Kaggle GPU memory.
N_FRAMES_PER_COMPLEX = 1    # fewer MD frames per complex to reduce memory
POCKET_CUTOFF = 6.0         # Angstrom, radius around ligand to define the pocket
MAX_COMPLEXES = None        # e.g. 100 for quick debugging; None = use all in split

BATCH_SIZE = 2              # smaller batch size to avoid CUDA OOM
NUM_EPOCHS = 50
LEARNING_RATE = 1e-3
ATOM_FEATURE_DIM = 128      # feature dim expected by pocket encoder (>= max atomic number)
CHECKPOINT_INTERVAL = 100   # save every N training steps

# Validation (early stopping), Option B: per-MD files only — train_test_val/MD_split_N_validate.txt.
# (Set USE_GLOBAL_VALIDATE_TXT True if you also have a global train_test_val/validate.txt.)
USE_GLOBAL_VALIDATE_TXT = False
GLOBAL_VALIDATE_TXT = os.path.join(RAW_DIR, 'train_test_val', 'validate.txt')
VAL_SPLIT_TYPE = 'val'
EARLY_STOP_PATIENCE = 15
EARLY_STOP_MIN_DELTA = 1e-6
MAX_VAL_COMPLEXES = None   # cap val pass for debugging; None = all val IDs in each file
BEST_MODEL_PATH = '/kaggle/working/multiscale_mdg_nn_best_val.pth'

# Outlier filtering: drop complexes whose selected-frame RMSD
# has any value outside IQR bounds computed from the train split.
ENABLE_OUTLIER_FILTER = True
APPLY_OUTLIER_FILTER_TO_VAL = True
OUTLIER_IQR_MULTIPLIER = 1.5

print('NOTE: This notebook infers MD structure from tiny_md.hdf5 in getting_started.ipynb.')


Imports done.
Using device: cuda
Found MD HDF5 files:
   /kaggle/input/datasets/uom210636r/misato/MD_split_1.hdf5
   /kaggle/input/datasets/uom210636r/misato/MD_split_10.hdf5
   /kaggle/input/datasets/uom210636r/misato/MD_split_2.hdf5
   /kaggle/input/datasets/uom210636r/misato/MD_split_3.hdf5
   /kaggle/input/datasets/uom210636r/misato/MD_split_4.hdf5
   /kaggle/input/datasets/uom210636r/misato/MD_split_5.hdf5
   /kaggle/input/datasets/uom210636r/misato/MD_split_6.hdf5
   /kaggle/input/datasets/uom210636r/misato/MD_split_7.hdf5
   /kaggle/input/datasets/uom210636r/misato/MD_split_8.hdf5
   /kaggle/input/datasets/uom210636r/misato/MD_split_9.hdf5
NOTE: This notebook infers MD structure from tiny_md.hdf5 in getting_started.ipynb.


In [5]:
# -------------------------------------------------------------------
# Utility functions (from TODOs 1–4)
# -------------------------------------------------------------------
from typing import List
def select_frame_indices(num_frames: int, n_select: int) -> List[int]:
    """Select roughly n_select frame indices across [0, num_frames-1]."""
    if n_select >= num_frames:
        return list(range(num_frames))
    idx = np.linspace(0, num_frames - 1, n_select, dtype=int)
    return sorted(set(idx.tolist()))

#Thilini

def load_split_ids(md_hdf5_path: str, split_type: str = "train") -> List[str]:
    """
    Load PDB IDs for a specific MD HDF5 file and split type.
    Args:
        md_hdf5_path: path to the MD HDF5 file (e.g., MD_split_1.hdf5)
        split_type: "train", "val", or "test"
    Returns:
        List of PDB IDs
    """
    # Extract base name of MD file, e.g., MD_split_1
    base_name = os.path.splitext(os.path.basename(md_hdf5_path))[0]
    
    # Construct path to the corresponding TXT file
    txt_file = os.path.join(
        RAW_DIR, "train_test_val", f"{base_name}_{split_type}.txt"
    )
    
    if not os.path.exists(txt_file):
        raise FileNotFoundError(f"No split file found for {base_name} {split_type}: {txt_file}")
    
    # Load PDB IDs from TXT
    with open(txt_file, "r") as f:
        ids = [line.strip() for line in f if line.strip()]
    
    print(f"Loaded {len(ids)} PDB IDs from {txt_file}")
    return ids

def build_residue_trajectories(
    misato_grp: h5py.Group,
    frame_indices: List[int],
) -> Tuple[Tensor, Tensor, np.ndarray]:
    """Build residue-level coordinates (static + MD trajectories) from atom trajectories.

    This version is streaming-friendly:
    - Reads frames one at a time from the HDF5 dataset instead of
      materializing all selected frames into memory.
    - Uses scatter-add + bincount style accumulation instead of
      per-residue boolean masks to compute centroids.
    """
    coords_ds = misato_grp['trajectory_coordinates']  # (T_full, A, 3)
    T_full, num_atoms, _ = coords_ds.shape
    T_sel = len(frame_indices)

    # Separate protein vs ligand via molecules_begin_atom_index (see getting_started).
    ligand_begin = misato_grp['molecules_begin_atom_index'][:][-1]
    protein_len = int(ligand_begin)

    residue_idx = misato_grp['atoms_residue'][:protein_len]  # (A_prot,)
    unique_res, inverse = np.unique(residue_idx, return_inverse=True)
    R = unique_res.shape[0]

    # Static residue centroids from the first selected frame.
    first_frame = frame_indices[0]
    coords0 = coords_ds[first_frame, :protein_len, :]  # (A_prot, 3)

    sums0 = np.zeros((R, 3), dtype=np.float32)
    counts0 = np.zeros(R, dtype=np.int64)
    np.add.at(sums0, inverse, coords0)
    np.add.at(counts0, inverse, 1)
    centroids0 = sums0 / counts0[:, None]

    # Build a simple (N, CA, C) triad around the centroid so that
    # ProteinGraphBuilder can compute orientation features.
    offset_N = np.array([0.5, 0.0, 0.0], dtype=np.float32)
    offset_C = np.array([0.0, 0.5, 0.0], dtype=np.float32)
    backbone_coords = np.stack(
        [
            centroids0 + offset_N,  # N-like point
            centroids0,             # CA-like point
            centroids0 + offset_C,  # C-like point
        ],
        axis=1,
    )  # (R, 3, 3)

    # Residue centroids over time (for dynamic correlation edges).
    md_res = np.zeros((T_sel, R, 3), dtype=np.float32)
    for ti, frame_idx in enumerate(frame_indices):
        coords_t = coords_ds[frame_idx, :protein_len, :]
        sums_t = np.zeros((R, 3), dtype=np.float32)
        counts_t = np.zeros(R, dtype=np.int64)
        np.add.at(sums_t, inverse, coords_t)
        np.add.at(counts_t, inverse, 1)
        centroids_t = sums_t / counts_t[:, None]
        md_res[ti] = centroids_t

    # Map global atom index -> residue index (protein atoms only; ligand set to -1).
    atom_to_residue_full = -np.ones(num_atoms, dtype=np.int64)
    atom_to_residue_full[:protein_len] = inverse

    backbone_coords_t = torch.from_numpy(backbone_coords).float()
    md_res_t = torch.from_numpy(md_res).float()
    return backbone_coords_t, md_res_t, atom_to_residue_full


def build_pocket_graphs_for_complex(
    misato_grp: h5py.Group,
    frame_indices: List[int],
    atom_to_residue_full: np.ndarray,
    pocket_builder: PocketGraphBuilder,
    pocket_cutoff: float,
    atom_feature_dim: int,
) -> Tuple[List[Data], Tensor]:
    """Build pocket atom graphs + stability labels (RMSD) for a single complex.

    - At each selected frame we:
      - Center on the ligand (defined by molecules_begin_atom_index and
        filtering out hydrogens via `atoms_number` == 1) and compute its centroid.
      - Take all atoms within `pocket_cutoff` Å of that centroid, unioned with
        all ligand heavy atoms.
      - Build one-hot atom features on atomic number and a boolean `is_ligand`
        mask, plus a per-atom `atom_to_residue` mapping for protein atoms.
      - Feed these into `PocketGraphBuilder` to get an atom-level graph.
    - For labels we take `frames_rmsd_ligand[frame_indices]` (RMSD per frame).
    """
    traj = misato_grp['trajectory_coordinates'][frame_indices, :, :]  # (T_sel, A, 3)
    T_sel, num_atoms, _ = traj.shape

    atom_numbers = misato_grp['atoms_number'][:]  # atomic numbers
    ligand_begin = misato_grp['molecules_begin_atom_index'][:][-1]

    # Ligand = last molecule, heavy atoms only.
    ligand_mask_global = np.arange(num_atoms) >= ligand_begin
    heavy_atom_mask = atom_numbers != 1
    ligand_mask_global = ligand_mask_global & heavy_atom_mask

    rmsd_all = misato_grp['frames_rmsd_ligand'][:]  # (T_full,)
    y_stability = torch.tensor(rmsd_all[frame_indices], dtype=torch.float32)

    pocket_graphs: List[Data] = []

    for ti in range(T_sel):
        coords_t = traj[ti]  # (A, 3)

        ligand_indices = np.where(ligand_mask_global)[0]
        if ligand_indices.size == 0:
            # Fallback: if ligand mask failed, treat all atoms as pocket.
            pocket_indices = np.arange(num_atoms)
        else:
            ligand_coords = coords_t[ligand_indices]
            ligand_centroid = ligand_coords.mean(axis=0, keepdims=True)
            dists_to_ligand = np.linalg.norm(coords_t - ligand_centroid, axis=1)
            pocket_mask = dists_to_ligand <= pocket_cutoff
            pocket_indices = np.where(pocket_mask | ligand_mask_global)[0]

        coords_pocket = torch.from_numpy(coords_t[pocket_indices].astype(np.float32))
        atom_nums_pocket = atom_numbers[pocket_indices].astype(np.int64)

        # Map atomic numbers 1..118 -> indices 0..117 for one-hot.
        atom_idx = atom_nums_pocket - 1
        atom_idx[atom_idx < 0] = 0
        atom_idx[atom_idx >= atom_feature_dim] = atom_feature_dim - 1
        atom_idx_t = torch.from_numpy(atom_idx)
        atom_features = torch.nn.functional.one_hot(
            atom_idx_t,
            num_classes=atom_feature_dim,
        ).float()

        atom_is_ligand_np = ligand_mask_global[pocket_indices]
        atom_is_ligand = torch.from_numpy(atom_is_ligand_np.astype(np.bool_))

        # Map pocket atoms to residue indices (protein only, ligand = -1).
        atom_to_residue = []
        for idx in pocket_indices:
            if ligand_mask_global[idx]:
                atom_to_residue.append(-1)
            else:
                atom_to_residue.append(int(atom_to_residue_full[idx]))
        atom_to_residue_t = torch.tensor(atom_to_residue, dtype=torch.long)

        pocket_inputs = PocketGraphInputs(
            atom_coords=coords_pocket,
            atom_features=atom_features,
            atom_is_ligand=atom_is_ligand,
            atom_to_residue=atom_to_residue_t,
        )
        pocket_data = pocket_builder(pocket_inputs)
        pocket_graphs.append(pocket_data)

    return pocket_graphs, y_stability


def validation_pdb_ids_for_md(md_path: str, file_pdb_ids: set) -> List[str]:
    """Val PDB IDs in this shard: per-MD validate list (Option B), or global validate.txt if USE_GLOBAL_VALIDATE_TXT."""
    if USE_GLOBAL_VALIDATE_TXT and os.path.isfile(GLOBAL_VALIDATE_TXT):
        ordered: List[str] = []
        with open(GLOBAL_VALIDATE_TXT, "r") as vf:
            for line in vf:
                pid = line.strip()
                if pid and pid in file_pdb_ids:
                    ordered.append(pid)
        return ordered
    base_name = os.path.splitext(os.path.basename(md_path))[0]
    txt_file = os.path.join(
        RAW_DIR, "train_test_val", f"{base_name}_{VAL_SPLIT_TYPE}.txt"
    )
    if not os.path.exists(txt_file):
        raise FileNotFoundError(f"No val split file: {txt_file}")
    with open(txt_file, "r") as tf:
        ids = [line.strip() for line in tf if line.strip()]
    return [p for p in ids if p in file_pdb_ids]


print('Utility functions defined.')


Utility functions defined.


In [6]:
# -------------------------------------------------------------------
# Inspect MD.hdf5 structure (via example complex)
# -------------------------------------------------------------------

if not MD_HDF5_FILES:
    raise FileNotFoundError(
        f'No MD*.hdf5 files found in {RAW_DIR}. Please check your MISATO raw directory.'
    )

md_example_path = MD_HDF5_FILES[0]
print('Inspecting MD file:', md_example_path)

with h5py.File(md_example_path, 'r') as f:
    pdb_ids = list(f.keys())
    print(f'Number of complexes in this MD file: {len(pdb_ids)}')
    if len(pdb_ids) == 0:
        raise RuntimeError('This MD.hdf5 appears to be empty.')

    first_id = pdb_ids[0]
    grp0 = f[first_id]
    print('First PDB ID:', first_id)
    print('Available datasets for first complex:')
    for k in grp0.keys():
        print('  ', k, grp0[k].shape)

    num_frames = grp0['trajectory_coordinates'].shape[0]
    print('Number of MD frames per complex:', num_frames)
    FRAME_INDICES = select_frame_indices(num_frames, N_FRAMES_PER_COMPLEX)
    print('Using frame indices (0-based):', FRAME_INDICES)


Inspecting MD file: /kaggle/input/datasets/uom210636r/misato/MD_split_1.hdf5
Number of complexes in this MD file: 1697
First PDB ID: 185L
Available datasets for first complex:
   atoms_element (2619,)
   atoms_number (2619,)
   atoms_residue (2619,)
   atoms_type (2619,)
   frames_bSASA (100,)
   frames_distance (100,)
   frames_interaction_energy (100,)
   frames_rmsd_ligand (100,)
   molecules_begin_atom_index (2,)
   trajectory_coordinates (100, 2619, 3)
Number of MD frames per complex: 100
Using frame indices (0-based): [0]


In [7]:
# -------------------------------------------------------------------
# Streaming pretraining on RMSD stability labels (no giant .pt file)
# with periodic checkpointing
# -------------------------------------------------------------------

from tqdm.auto import tqdm

TXT_SPLIT_DIR = '/kaggle/input/datasets/uom210636r/misato/train_test_val/'

# Instantiate model and trainer once.
model = MultiscaleMDGNN(atom_feature_dim=ATOM_FEATURE_DIM)
model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
trainer = Trainer(model=model, optimizer=optimizer, lambda_temp=0.1, alpha_multitask=0.0, device=device)

# Optionally resume from an existing checkpoint.
start_epoch = 1
global_step = 0
best_val_rmse = float('inf')
early_stop_counter = 0
if os.path.exists(LOAD_CHECKPOINT_PATH):
    print(f'Found checkpoint at {LOAD_CHECKPOINT_PATH}, loading...')
    ckpt = torch.load(LOAD_CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    start_epoch = ckpt.get('epoch', 1)
    global_step = ckpt.get('global_step', 0)
    best_val_rmse = ckpt.get('best_val_rmse', float('inf'))
    early_stop_counter = ckpt.get('early_stop_counter', 0)
    print(f'Resuming from epoch {start_epoch}, global_step {global_step}.')

if USE_GLOBAL_VALIDATE_TXT and os.path.isfile(GLOBAL_VALIDATE_TXT):
    print('Validation split: global', GLOBAL_VALIDATE_TXT)
else:
    print(f'Validation split (Option B): per-MD train_test_val/MD_*_{VAL_SPLIT_TYPE}.txt')


def run_validation_epoch() -> Tuple[float, int]:
    """Root-mean-square error on ligand RMSD (same frame setup as training)."""
    model.eval()
    se_sum = 0.0
    n_pred = 0
    skipped_val_outlier = 0

    def _accumulate_val_batch(items: List[Dict[str, Any]]) -> None:
        nonlocal se_sum, n_pred
        if not items:
            return
        num_frames_selected = len(items[0]['pocket_graphs'])
        for t_local in range(num_frames_selected):
            protein_graphs = []
            pocket_graphs_t = []
            y_list = []
            for it in items:
                protein_graphs.append(it['protein_graph'])
                pocket_graphs_t.append(it['pocket_graphs'][t_local])
                y_list.append(it['y_stability'][t_local])
            y_t = torch.stack(y_list, dim=0).float()
            gbatch = collate_complexes(
                protein_graphs,
                pocket_graphs_t,
                {'y_stability': y_t},
            )
            out = model(
                {
                    'protein': gbatch.protein.to(device),
                    'pocket': gbatch.pocket.to(device),
                }
            )
            pred = out['y_pred'].view(-1)
            y_true = y_t.to(device)
            se_sum += torch.sum((pred - y_true) ** 2).item()
            n_pred += pred.numel()

    with torch.no_grad():
        for md_path in MD_HDF5_FILES:
            protein_builder = ProteinGraphBuilder()
            corr_builder = CorrelationEdgeBuilder()
            pocket_builder = PocketGraphBuilder()
            with h5py.File(md_path, 'r') as f:
                file_pdb_ids = set(f.keys())
                val_ids = validation_pdb_ids_for_md(md_path, file_pdb_ids)
                if MAX_VAL_COMPLEXES is not None:
                    val_ids = val_ids[:MAX_VAL_COMPLEXES]
                pending_v: List[Dict[str, Any]] = []
                for pdb_id in tqdm(
                    val_ids,
                    desc=f'Val {os.path.basename(md_path)}',
                    leave=False,
                ):
                    if pdb_id not in file_pdb_ids:
                        continue
                    grp = f[pdb_id]

                    if APPLY_OUTLIER_FILTER_TO_VAL and outlier_bounds is not None:
                        low, high = outlier_bounds
                        rmsd_sel = grp['frames_rmsd_ligand'][FRAME_INDICES]
                        if np.any((rmsd_sel < low) | (rmsd_sel > high)):
                            skipped_val_outlier += 1
                            continue

                    backbone_coords, md_res_coords, atom_to_residue_full = (
                        build_residue_trajectories(grp, FRAME_INDICES)
                    )
                    protein_inputs = ProteinGraphInputs(
                        backbone_coords=backbone_coords,
                        md_residue_coords=md_res_coords,
                    )
                    protein_graph = protein_builder(protein_inputs)
                    protein_graph = corr_builder.add_correlation_edges(
                        protein_graph, md_res_coords
                    )
                    pocket_graphs, y_stability = build_pocket_graphs_for_complex(
                        grp,
                        FRAME_INDICES,
                        atom_to_residue_full,
                        pocket_builder=pocket_builder,
                        pocket_cutoff=POCKET_CUTOFF,
                        atom_feature_dim=ATOM_FEATURE_DIM,
                    )
                    pending_v.append(
                        {
                            'protein_graph': protein_graph,
                            'pocket_graphs': pocket_graphs,
                            'y_stability': y_stability,
                        }
                    )
                    if len(pending_v) >= BATCH_SIZE:
                        _accumulate_val_batch(pending_v)
                        pending_v = []
                        if torch.cuda.is_available():
                            torch.cuda.empty_cache()
                _accumulate_val_batch(pending_v)

    model.train()
    if n_pred == 0:
        return float('nan'), skipped_val_outlier
    return float(math.sqrt(se_sum / n_pred)), skipped_val_outlier


# Precompute train-split RMSD outlier bounds once (same FRAME_INDICES as training).
outlier_bounds = None
if ENABLE_OUTLIER_FILTER:
    print('Computing train RMSD outlier bounds (IQR)...')
    train_rmsd_values: List[float] = []
    for md_path in MD_HDF5_FILES:
        train_ids = load_split_ids(md_path, split_type='train')
        if MAX_COMPLEXES is not None:
            train_ids = train_ids[:MAX_COMPLEXES]
        with h5py.File(md_path, 'r') as f:
            file_pdb_ids = set(f.keys())
            for pdb_id in train_ids:
                if pdb_id not in file_pdb_ids:
                    continue
                grp = f[pdb_id]
                rmsd_sel = grp['frames_rmsd_ligand'][FRAME_INDICES]
                train_rmsd_values.extend(rmsd_sel.astype(np.float64).tolist())

    if len(train_rmsd_values) >= 8:
        vals = np.asarray(train_rmsd_values, dtype=np.float64)
        q1 = float(np.percentile(vals, 25.0))
        q3 = float(np.percentile(vals, 75.0))
        iqr = q3 - q1
        low = max(0.0, q1 - OUTLIER_IQR_MULTIPLIER * iqr)
        high = q3 + OUTLIER_IQR_MULTIPLIER * iqr
        outlier_bounds = (low, high)
        print(f'Outlier bounds: [{low:.4f}, {high:.4f}] from {len(vals)} train frame labels.')
    else:
        print('Not enough train RMSD values for IQR bounds; outlier filtering disabled.')

print('Starting streaming pretraining...')

for epoch in range(start_epoch, NUM_EPOCHS + 1):
    epoch_losses: List[float] = []
    skipped_outlier = 0
    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")

    # Iterate over all MD files for this epoch.
    for md_path in MD_HDF5_FILES:
        # Load the correct train IDs for this MD file
        train_ids = load_split_ids(md_path, split_type="train")
        if MAX_COMPLEXES is not None:
            train_ids = train_ids[:MAX_COMPLEXES]
            print(f'Restricting to first {len(train_ids)} complexes for {os.path.basename(md_path)}.')

        # Shuffle complexes within this file each epoch.
        rng = np.random.default_rng()
        train_ids = list(train_ids)
        rng.shuffle(train_ids)

        protein_builder = ProteinGraphBuilder()
        corr_builder = CorrelationEdgeBuilder()
        pocket_builder = PocketGraphBuilder()

        with h5py.File(md_path, 'r') as f:
            file_pdb_ids = set(f.keys())

            # Accumulate complexes into a small in-memory batch, then train and discard.
            pending_complexes: List[Dict[str, Any]] = []

            for pdb_id in tqdm(train_ids, desc=f'Epoch {epoch} - {os.path.basename(md_path)}', leave=False):
                if pdb_id not in file_pdb_ids:
                    continue

                grp = f[pdb_id]

                # Optional outlier removal before graph building.
                if outlier_bounds is not None:
                    low, high = outlier_bounds
                    rmsd_sel = grp['frames_rmsd_ligand'][FRAME_INDICES]
                    if np.any((rmsd_sel < low) | (rmsd_sel > high)):
                        skipped_outlier += 1
                        continue

                # Build residue-level trajectories (protein graph inputs).
                backbone_coords, md_res_coords, atom_to_residue_full = build_residue_trajectories(
                    grp,
                    FRAME_INDICES,
                )

                protein_inputs = ProteinGraphInputs(
                    backbone_coords=backbone_coords,
                    md_residue_coords=md_res_coords,
                )
                protein_graph = protein_builder(protein_inputs)
                protein_graph = corr_builder.add_correlation_edges(protein_graph, md_res_coords)

                # Build pocket graphs and labels for this complex.
                pocket_graphs, y_stability = build_pocket_graphs_for_complex(
                    grp,
                    FRAME_INDICES,
                    atom_to_residue_full,
                    pocket_builder=pocket_builder,
                    pocket_cutoff=POCKET_CUTOFF,
                    atom_feature_dim=ATOM_FEATURE_DIM,
                )

                pending_complexes.append({
                    'protein_graph': protein_graph,
                    'pocket_graphs': pocket_graphs,
                    'y_stability': y_stability,
                })

                # When we have enough complexes for a batch, run one training step.
                if len(pending_complexes) >= BATCH_SIZE:
                    num_frames_selected = len(pending_complexes[0]['pocket_graphs'])
                    frame_batches = []

                    for t_local in range(num_frames_selected):
                        protein_graphs = []
                        pocket_graphs_t = []
                        y_list = []

                        for item in pending_complexes:
                            protein_graphs.append(item['protein_graph'])
                            pocket_graphs_t.append(item['pocket_graphs'][t_local])
                            y_list.append(item['y_stability'][t_local])

                        labels = {
                            'y_stability': torch.stack(y_list, dim=0).float(),
                        }
                        gbatch = collate_complexes(protein_graphs, pocket_graphs_t, labels)
                        frame_batches.append(gbatch)

                    loss_dict = trainer.pretrain_step(frame_batches)
                    epoch_losses.append(loss_dict['loss'].item())

                    global_step += 1
                    # Periodic checkpointing.
                    if global_step % CHECKPOINT_INTERVAL == 0:
                        torch.save(
                            {
                                'epoch': epoch,
                                'global_step': global_step,
                                'model_state_dict': model.state_dict(),
                                'optimizer_state_dict': optimizer.state_dict(),
                                'best_val_rmse': best_val_rmse,
                                'early_stop_counter': early_stop_counter,
                            },
                            CHECKPOINT_PATH,
                        )
                        print(f'Saved checkpoint at step {global_step} to {CHECKPOINT_PATH}.')

                    # Discard the batch from memory.
                    pending_complexes = []

            # Handle any leftover complexes in this file (< BATCH_SIZE).
            if pending_complexes:
                num_frames_selected = len(pending_complexes[0]['pocket_graphs'])
                frame_batches = []

                for t_local in range(num_frames_selected):
                    protein_graphs = []
                    pocket_graphs_t = []
                    y_list = []

                    for item in pending_complexes:
                        protein_graphs.append(item['protein_graph'])
                        pocket_graphs_t.append(item['pocket_graphs'][t_local])
                        y_list.append(item['y_stability'][t_local])

                    labels = {
                        'y_stability': torch.stack(y_list, dim=0).float(),
                    }
                    gbatch = collate_complexes(protein_graphs, pocket_graphs_t, labels)
                    frame_batches.append(gbatch)

                loss_dict = trainer.pretrain_step(frame_batches)
                epoch_losses.append(loss_dict['loss'].item())

                global_step += 1
                if global_step % CHECKPOINT_INTERVAL == 0:
                    torch.save(
                        {
                            'epoch': epoch,
                            'global_step': global_step,
                            'model_state_dict': model.state_dict(),
                            'optimizer_state_dict': optimizer.state_dict(),
                            'best_val_rmse': best_val_rmse,
                            'early_stop_counter': early_stop_counter,
                        },
                        CHECKPOINT_PATH,
                    )
                    print(f'Saved checkpoint at step {global_step} to {CHECKPOINT_PATH}.')

                pending_complexes = []

    mean_loss = float(np.mean(epoch_losses)) if epoch_losses else float('nan')
    print(f'Epoch {epoch}/{NUM_EPOCHS} - mean train loss: {mean_loss:.4f}')
    if outlier_bounds is not None:
        print(f'Epoch {epoch}/{NUM_EPOCHS} - skipped outlier complexes: {skipped_outlier}')

    val_rmse, skipped_val_outlier = run_validation_epoch()
    print(f'Epoch {epoch}/{NUM_EPOCHS} - val RMSE (ligand RMSD): {val_rmse:.6f}')
    if APPLY_OUTLIER_FILTER_TO_VAL and outlier_bounds is not None:
        print(f'Epoch {epoch}/{NUM_EPOCHS} - skipped val outlier complexes: {skipped_val_outlier}')

    if math.isnan(val_rmse):
        print('  No val predictions (empty split?); early stopping unchanged.')
    elif val_rmse < best_val_rmse - EARLY_STOP_MIN_DELTA:
        best_val_rmse = val_rmse
        early_stop_counter = 0
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f'  New best val RMSE -> {BEST_MODEL_PATH}')
    else:
        early_stop_counter += 1
        print(
            f'  No val improvement ({early_stop_counter}/{EARLY_STOP_PATIENCE} patience).'
        )

    if early_stop_counter >= EARLY_STOP_PATIENCE:
        print('Early stopping: reverting patience criterion met.')
        break

print('Streaming pretraining complete.')

# -------------------------------------------------------------------
# Save final model (best val weights when available)
# -------------------------------------------------------------------

if os.path.isfile(BEST_MODEL_PATH):
    model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
    print('Loaded best val weights from:', BEST_MODEL_PATH)
else:
    print('No best val checkpoint on disk; saving last epoch weights.')

torch.save(model.state_dict(), MODEL_PATH)
print('Saved pretrained model to:', MODEL_PATH)

# Quick sanity-check prediction on one example complex/frame.
example_protein_graph = None
example_pocket_graph = None
example_y = None

if MD_HDF5_FILES:
    md_example_path = MD_HDF5_FILES[0]
    train_ids_example = load_split_ids(md_example_path, split_type="train")
    if train_ids_example:
        with h5py.File(md_example_path, 'r') as f:
            for pdb_id in train_ids_example:
                if pdb_id not in f:
                    continue
                grp = f[pdb_id]
                backbone_coords, md_res_coords, atom_to_residue_full = build_residue_trajectories(
                    grp,
                    FRAME_INDICES,
                )
                protein_inputs = ProteinGraphInputs(
                    backbone_coords=backbone_coords,
                    md_residue_coords=md_res_coords,
                )
                protein_graph = ProteinGraphBuilder()(protein_inputs)
                protein_graph = CorrelationEdgeBuilder().add_correlation_edges(protein_graph, md_res_coords)
                pocket_graphs, y_stability = build_pocket_graphs_for_complex(
                    grp,
                    FRAME_INDICES,
                    atom_to_residue_full,
                    pocket_builder=PocketGraphBuilder(),
                    pocket_cutoff=POCKET_CUTOFF,
                    atom_feature_dim=ATOM_FEATURE_DIM,
                )
                example_protein_graph = protein_graph
                example_pocket_graph = pocket_graphs[0]
                example_y = y_stability[0:1]
                break

if example_protein_graph is not None and example_pocket_graph is not None:
    gbatch_example = collate_complexes(
        [example_protein_graph],
        [example_pocket_graph],
        labels={'y_stability': example_y},
    )
    model.eval()
    with torch.no_grad():
        out = model(
            {
                'protein': gbatch_example.protein.to(device),
                'pocket': gbatch_example.pocket.to(device),
            }
        )
        print('Example prediction (first complex, first frame):',
              out['y_pred'].cpu().numpy().ravel())
        print('True RMSD:', float(example_y[0]))
else:
    print('Could not find an example complex for sanity-check prediction.')


Found checkpoint at /kaggle/working/project/output/multiscale_mdg_nn_checkpoint.pth, loading...
Resuming from epoch 35, global_step 191600.
Validation split (Option B): per-MD train_test_val/MD_*_val.txt
Computing train RMSD outlier bounds (IQR)...
Loaded 1187 PDB IDs from /kaggle/input/datasets/uom210636r/misato/train_test_val/MD_split_1_train.txt
Loaded 1189 PDB IDs from /kaggle/input/datasets/uom210636r/misato/train_test_val/MD_split_10_train.txt
Loaded 1187 PDB IDs from /kaggle/input/datasets/uom210636r/misato/train_test_val/MD_split_2_train.txt
Loaded 1187 PDB IDs from /kaggle/input/datasets/uom210636r/misato/train_test_val/MD_split_3_train.txt
Loaded 1187 PDB IDs from /kaggle/input/datasets/uom210636r/misato/train_test_val/MD_split_4_train.txt
Loaded 1187 PDB IDs from /kaggle/input/datasets/uom210636r/misato/train_test_val/MD_split_5_train.txt
Loaded 1187 PDB IDs from /kaggle/input/datasets/uom210636r/misato/train_test_val/MD_split_6_train.txt
Loaded 1187 PDB IDs from /kaggle/inp

Epoch 35 - MD_split_1.hdf5:   0%|          | 0/1187 [00:00<?, ?it/s]

Saved checkpoint at step 191700 to /kaggle/working/multiscale_mdg_nn_checkpoint.pth.
Saved checkpoint at step 191800 to /kaggle/working/multiscale_mdg_nn_checkpoint.pth.
Saved checkpoint at step 191900 to /kaggle/working/multiscale_mdg_nn_checkpoint.pth.
Saved checkpoint at step 192000 to /kaggle/working/multiscale_mdg_nn_checkpoint.pth.
Saved checkpoint at step 192100 to /kaggle/working/multiscale_mdg_nn_checkpoint.pth.
Loaded 1189 PDB IDs from /kaggle/input/datasets/uom210636r/misato/train_test_val/MD_split_10_train.txt


Epoch 35 - MD_split_10.hdf5:   0%|          | 0/1189 [00:00<?, ?it/s]

Saved checkpoint at step 192200 to /kaggle/working/multiscale_mdg_nn_checkpoint.pth.
Saved checkpoint at step 192300 to /kaggle/working/multiscale_mdg_nn_checkpoint.pth.
Saved checkpoint at step 192400 to /kaggle/working/multiscale_mdg_nn_checkpoint.pth.
Saved checkpoint at step 192500 to /kaggle/working/multiscale_mdg_nn_checkpoint.pth.
Saved checkpoint at step 192600 to /kaggle/working/multiscale_mdg_nn_checkpoint.pth.
Saved checkpoint at step 192700 to /kaggle/working/multiscale_mdg_nn_checkpoint.pth.
Loaded 1187 PDB IDs from /kaggle/input/datasets/uom210636r/misato/train_test_val/MD_split_2_train.txt


...

Val MD_split_6.hdf5:   0%|          | 0/255 [00:00<?, ?it/s]

Val MD_split_7.hdf5:   0%|          | 0/255 [00:00<?, ?it/s]

Val MD_split_8.hdf5:   0%|          | 0/255 [00:00<?, ?it/s]

Val MD_split_9.hdf5:   0%|          | 0/255 [00:00<?, ?it/s]

Epoch 50/50 - val RMSE (ligand RMSD): 1.103690
Epoch 50/50 - skipped val outlier complexes: 168
  No val improvement (2/15 patience).
Streaming pretraining complete.
Loaded best val weights from: /kaggle/working/multiscale_mdg_nn_best_val.pth
Saved pretrained model to: /kaggle/working/multiscale_mdg_nn_pretrained.pth
Loaded 1187 PDB IDs from /kaggle/input/datasets/uom210636r/misato/train_test_val/MD_split_1_train.txt
Example prediction (first complex, first frame): [2.3593826]
True RMSD: 0.9518974423408508


In [8]:
# This cell is now deprecated because pretraining is done in the previous
# cell in a streaming fashion (no large .pt file).
print('Pretraining is handled in the previous cell via streaming. Nothing to do here.')


Pretraining is handled in the previous cell via streaming. Nothing to do here.


In [9]:
# Model saving and sanity-check are now part of the streaming
# pretraining cell (previous cell). This cell is kept only to
# avoid breaking execution order.
print('Model has already been saved and sanity-checked in the previous cell.')


Model has already been saved and sanity-checked in the previous cell.
